In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "pysmartdata2026";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_smartdata`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_pysmartdata CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_pysmartdata
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS catalog_pysmartdata.raw;
DROP SCHEMA IF EXISTS catalog_pysmartdata.bronze;
DROP SCHEMA IF EXISTS catalog_pysmartdata.silver;
DROP SCHEMA IF EXISTS catalog_pysmartdata.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_pysmartdata.raw;
CREATE SCHEMA IF NOT EXISTS catalog_pysmartdata.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_pysmartdata.silver;
CREATE SCHEMA IF NOT EXISTS catalog_pysmartdata.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pysmartdata.bronze.facturacion (
periodo integer,
suscriptor integer,
nombre_cliente string,
direccion string, 
estrato INTEGER,
codigo_concepto INTEGER,
concepto string,
consumo_m3 integer, 
valor_cargo_fijo double, 
valor_consumo double,   
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/facturacion"

In [0]:

CREATE TABLE IF NOT EXISTS catalog_pysmartdata.bronze.recaudos (
suscriptor integer,
periodo integer,
entidad string,
descripcion string, 
codigo_punto_recaudo integer,
descripcion_punto_recaudo string,
fecha_pago date, 
codigo_tarifa integer,
codigo_concepto integer, 
concepto string ,  
valor_pagado double,    
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/recaudos"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pysmartdata.silver.facturacion_transformed (
periodo integer,
suscriptor integer,
nombre_cliente string,
direccion string, 
estrato INTEGER,
codigo_concepto INTEGER,
concepto string,
consumo_m3 integer, 
valor_cargo_fijo double, 
valor_consumo double,   
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/facturacion_transformed"

In [0]:

CREATE TABLE IF NOT EXISTS catalog_pysmartdata.silver.recaudos_transformed (
suscriptor integer,
periodo integer,
entidad string,
descripcion string, 
codigo_punto_recaudo integer,
descripcion_punto_recaudo string,
fecha_pago date, 
codigo_tarifa integer,
codigo_concepto integer, 
concepto string ,  
valor_pagado double,    
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/recaudos_transformed"

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.golden_raced_partitioned (
  race_year integer,
  conteo long,
  max_altitude integer,
  min_altitude integer,
  country string,
  race_type string,
  near_equator string
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_raced_partitioned"